In [ ]:
from pathlib import Path

import torch
import torch.nn.functional as F
import torch.nn as nn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import ml_tools as mlt
import sim_ranking as sr

In [ ]:
base_result_dir = Path("/Users/claudy/dev/work/data/sim_ranking/results/gnn/1004_0826_cv_nzgmdb3p4_4_8_normIMs_2000epochs")

In [ ]:
run_config = sr.ml.gnn_gm.RunConfig.from_yaml(base_result_dir / "run_config.yaml")

In [ ]:
ims = run_config.ims
pred_keys = mlt.array_utils.numpy_str_join("_", ims, "pred")
pred_std_keys = mlt.array_utils.numpy_str_join("_", ims, "pred_std")

In [ ]:
def run_for_cv(result_dir: Path):
	print(f"------------ {result_dir.stem} --------------")
	val_df = pd.read_parquet(result_dir / "val_results.parquet")
	train_df = pd.read_parquet(result_dir / "train_results.parquet")
    
	val_pred_mean = torch.from_numpy(val_df[pred_keys].values)
	val_pred_std = torch.from_numpy(val_df[pred_std_keys].values) 
	val_target = torch.from_numpy(val_df[ims].values)
	
	val_mask = ~torch.isnan(val_target)
	
	val_nll = F.gaussian_nll_loss(val_pred_mean[val_mask], val_target[val_mask], val_pred_std[val_mask]**2)
	val_mse = F.mse_loss(val_pred_mean[val_mask], val_target[val_mask])
	mean_sigma = val_pred_std[val_mask].mean()
	print(f"NLL val: {val_nll}, MSE val: {val_mse}, mean sigma: {mean_sigma}")
	
	train_pred_mean = torch.from_numpy(train_df[pred_keys].values)
	train_pred_std = torch.from_numpy(train_df[pred_std_keys].values)
	train_target = torch.from_numpy(train_df[ims].values)
	
	train_mask = ~torch.isnan(train_target)
	
	train_nll = F.gaussian_nll_loss(train_pred_mean[train_mask], train_target[train_mask], train_pred_std[train_mask]**2)
	train_mse = F.mse_loss(train_pred_mean[train_mask], train_target[train_mask])
	mean_sigma = train_pred_std[train_mask].mean()
	
	print(f"NLL train: {train_nll}, MSE train: {train_mse}, mean sigma: {mean_sigma}")

In [ ]:
cv_dirs = [cur_dir for cur_dir in base_result_dir.iterdir() if cur_dir.is_dir() and cur_dir.name.startswith("cv")]
for cur_dir in cv_dirs:
    	run_for_cv(cur_dir)